# Prétraitement de la base E2 : construction de `Base1Base2_clean`

Ce notebook définit une fonction de prétraitement `load_and_clean(path)` qui 
prépare la base de données utilisée dans les analyses descriptives et la 
modélisation de topics (BERTopic).

Principales opérations :
- chargement de la base brute `Base1Base2.csv` ;
- suppression des colonnes techniques du questionnaire (date, langue, etc.) ;
- regroupement des variables multi-réponses en colonnes synthétiques 
  (ex. activité professionnelle, psychothérapies) ;
- standardisation de certaines valeurs textuelles (accents, formulations) ;
- création d'une base nettoyée prête pour l'analyse.

La base nettoyée produite par `load_and_clean` est ensuite sauvegardée et 
réutilisée dans les notebooks :

- `Analyse descriptive E2.ipynb`  
- `BERTopic E2.ipynb`  
- `Wordcloud.E2.ipynb`

In [ ]:
import pandas as pd
import unicodedata
import numpy as np

def load_and_clean(path):

    """
    Charge et nettoie la base brute E2 (`Base1Base2.csv`).

    Étapes principales :
    - lecture du fichier CSV brut ;
    - suppression des colonnes techniques (date de soumission, dernière page,
      langue, consentement, tête de série) ;
    - regroupement des colonnes liées à l'activité professionnelle en une 
      colonne synthétique (ex. "Activité professionnelle") ;
    - regroupement des colonnes liées aux psychothérapies en une colonne 
      synthétique (ex. "Psychothérapies") ;
    - harmonisation / standardisation des valeurs textuelles lorsque nécessaire 
      (ex. accents, variantes d'écriture) ;
    - préparation d'une base prête à être utilisée pour les analyses 
      descriptives et la modélisation de topics.

    Paramètres
    ----------
    path : str
        Chemin vers le fichier CSV brut (par ex. "Data/Base1Base2.csv").

    Retour
    ------
    df : pandas.DataFrame
        Base nettoyée et structurée, prête pour les analyses suivantes.
    """

    df = pd.read_csv(path, sep = ';')

        # --- Suppression des colonnes techniques du questionnaire ---
    df.drop(columns=['Date de soumission'], inplace=True)
    df.drop(columns=['Dernière page'], inplace=True)
    df.drop(columns=['Langue de départ'], inplace=True)
    df.drop(columns=["J'accepte"], inplace=True)
    df.drop(columns=["Tête de série"], inplace=True)

    # --- Construction de la variable "Activité professionnelle" ---
    # 1) Sélection des colonnes d'activité pro (standard + [Autre])
    # 2) Extraction des libellés depuis les noms de colonnes
    # 3) Fusion des colonnes en une colonne textuelle unique
    # 4) Suppression des colonnes d'origine

    # --- Construction de la variable "Psychothérapies" ---
    # 1) Sélection des colonnes contenant les réponses aux psychothérapies
    # 2) Fusion des réponses cochées en une chaîne de type "TCC, Thérapie de soutien"
    # 3) Renommage éventuel de la colonne [Autre] et suppression des colonnes binaires

    # --- Autres opérations de nettoyage / transformation ---
    # (Standardisation d'accents, harmonisation de certaines modalités, etc.)


    # Crée une colonne rassemblant les différentes activité professionnelles

        # Sélection des colonnes contenant les réponses aux psychothérapies
    colonnes_activite = [col for col in df.columns if "Quelle est votre activité professionnelle" in col]
    
    # Détection de la colonne [Autre]
    cola_autre = [col for col in colonnes_activite if "[Autre]" in col]
    
    # Colonnes à regrouper (sans [Autre])
    colonnes_activite_sans_autre = [col for col in colonnes_activite if col not in cola_autre]
    
    # Création de la colonne 
    df["activité professionnelle"] = df[colonnes_activite_sans_autre].apply(
        lambda row: ", ".join(
            [col.split("[")[-1].strip("]") for col in colonnes_activite_sans_autre if str(row[col]).strip().lower() == "oui"]
        ),
        axis=1
    )

    
    # Renommer la colonne [Autre] pour qu’elle soit plus lisible
    if cola_autre:
        df.rename(columns={cola_autre[0]: "Activité Professionnelle - Autre"}, inplace=True)
    
    def fusion_repondant(row):
        std = str(row["activité professionnelle"]).strip() if pd.notna(row["activité professionnelle"]) else ""
        autre = str(row["Activité Professionnelle - Autre"]).strip() if pd.notna(row["Activité Professionnelle - Autre"]) else ""
        if std and autre:
            return f"{std}, {autre}"
        return std or autre or "Non renseigné"
    
    # Création des colonnes fusionnées
    df["Activité Professionnelle"] = df.apply(fusion_repondant, axis=1)
    
    # Suppression des colonnes d'origine
    df.drop(columns=["activité professionnelle", "Activité Professionnelle - Autre"] + colonnes_activite_sans_autre , inplace=True)




    # Création d'une colonne rassemblant les différentes psychothérapies
        # Sélection des colonnes contenant les réponses aux psychothérapies
    colonnes_psychotherapie = [col for col in df.columns if "Merci d'indiquer ici la ou les psychothérapie(s)" in col]
    col_autre = [col for col in colonnes_psychotherapie if "[Autre]" in col]

    colonnes_psychotherapie_sans_autre = [col for col in colonnes_psychotherapie if col not in col_autre]
    
    # Création de la colonne "Psychothérapies"
    df["psychothérapies"] = df[colonnes_psychotherapie_sans_autre].apply(
        lambda row: ", ".join(
            [col.split("[")[-1].strip("]") for col in colonnes_psychotherapie_sans_autre if str(row[col]).strip().lower() == "oui"]
        ),
        axis=1
    )
    
    # Renommer la colonne [Autre] pour qu’elle soit plus lisible
    if col_autre:
        df.rename(columns={col_autre[0]: "Psychothérapie - Autre"}, inplace=True)

    
    def fusion_psy(row):
        std = str(row["psychothérapies"]).strip() if pd.notna(row["psychothérapies"]) else ""
        autre = str(row["Psychothérapie - Autre"]).strip() if pd.notna(row["Psychothérapie - Autre"]) else ""
        if std and autre:
            return f"{std}, {autre}"
        return std or autre or "Non renseigné"
    
    # Création des colonnes fusionnées
    df["Psychothérapie"] = df.apply(fusion_psy, axis=1)
    
    # Suppression des colonnes d'origine
    df.drop(columns=["psychothérapies", "Psychothérapie - Autre"] + colonnes_psychotherapie_sans_autre, inplace=True)


    # Création d'une colonne regroupant les troubles

    col_diag = [col for col in df.columns if "psychiatriques" in col]
    
    # Fonction pour supprimer les accents
    def remove_accents(text):
        return ''.join(
            c for c in unicodedata.normalize('NFD', text)
            if unicodedata.category(c) != 'Mn'
        )
    
    # Fonction de classification
    def classer_troubles(texte):
        if pd.isna(texte):
            return np.nan
    
        # Nettoyage du texte
        texte = texte.lower()
        texte = remove_accents(texte)
    
        categories = set()
    
        if any(mot in texte for mot in ["depression", "depressive", "deprime", "depressif", "ts", "edc"]):
            categories.add("Depression")
    
        if any(mot in texte for mot in ["anxiete", "angoisse", "anxiete generalisee", "stress", "anxieux", "tag"]):
            categories.add("Anxiety")
    
        if any(mot in texte for mot in ["bipolaire","bipolarite", "bi polarité", "bi polaire", "bi-polarité", "pmd"]):
            categories.add("Bipolar disorder")
    
        if any(mot in texte for mot in ["alcool", "alcoolisme", "alcoolique", "addiction", "addictions", "dependance", "drogue", "toxicomanie",
                                        "tabac", "tabagique"
                                       ]):
            categories.add("Addiction disorder")
    
        if any(mot in texte for mot in ["borderline", "personnalite", "manipule", "manipulateur", "trouble de perso"]):
            categories.add("Personality disorder")
    
        if any(mot in texte for mot in ["boulimie", "anorexie", "alimentaire", "alimentation", "tca"]):
            categories.add("Eating disorder")
    
        if any(mot in texte for mot in ["alzheimer", "declin", "memoire", "demence", "cognitif", "desorientation", "confusion",
                                       "parkinson", "ecriture et lecture lentes"]):
            categories.add("Cognitive disorder")
    
        if any(mot in texte for mot in [
            "schizophrenie", "schizophrène", "hallucination", "hallucinations", 
            "delire", "delires", "paranoia", "paranoiaque","paranoïde", "psychose", 
            "psychotique", "trouble psychotique", "dissociation", "dissociatif"
        ]):
            categories.add("Psychotic disorder")
    
        # Cas spéciaux ou flous
        if any(mot in texte for mot in [
            "sspt", "ptsd", "trauma", "toc", "hyperactivite", "insecurite", "trouble", "instabilite", 
            "dyspraxie", "hypersensibilite", "trouble obsessionnel compulsif", "lassitude", "burn out",
            "burnout"
        ]):
            categories.add("Other psychiatric disorder")
    
        if not categories:
            return "Other psychiatric disorder"
    
        return ", ".join(sorted(categories))
    
    
    # Application sur une colonne de texte libre
    df["Trouble_Catégorisé"] = df[col_diag[0]].apply(classer_troubles)
    df["Trouble_Passé_Catégorisé"] = df[col_diag[1]].apply(classer_troubles)

    return df



## 2. Application du prétraitement et vérifications de base

Nous appliquons maintenant la fonction `load_and_clean` au fichier brut 
`Base1Base2.csv` pour obtenir la base nettoyée.  
Nous affichons ensuite les premières lignes pour vérifier le résultat.

In [ ]:
# Application de la fonction de prétraitement à la base brute
df = load_and_clean("Data/Base1Base2.csv")

# Aperçu des premières lignes
df.head()

,ID de la réponse,Genre,Genre [Autre],Age,Quel est votre niveau d'étude ?,"Merci de nous indiquez ici le ou les trouble(s) psychiatriques (e.g. dépression) dont vous souffrez actuellement (si vous ne souffrez pas de trouble psychiatrique, passez à la question suivante)",Ce(s) trouble(s) ont-il été diagnostiqué(s) par un psychiatre?,"Merci d'indiquer ici le ou les troubles psychiatriques dont vous avez pu souffrir par le passé (si vous n'avez pas d'antécédent de trouble psychiatrique, passez à la question suivante)","Au quotidien, quelles sont les pensées (ex : « Je suis nul(le) ») qui peuvent éventuellement vous faire souffrir, pouvez-vous nous les expliquer ?","Au quotidien, quelles sont les émotions (ex : tristesse ») qui peuvent éventuellement vous faire souffrir, pouvez-vous nous les expliquer ?",...,Que voudriez-vous voir changer dans votre quotidien ?,"Si vous avez déjà suivi une psychothérapie, que vous a-t-elle apportée ?","Si vous avez bénéficié de psychothérapie, qu’est ce qui a été travaillé avec votre thérapeute ?","Qu’est-ce qui peut, selon vous, constituer un frein à l’amélioration de votre état ?",Commentaire libre,Diagnostic_mots_cles,Activité Professionnelle,Psychothérapie,Trouble_Catégorisé,Trouble_Passé_Catégorisé
0,11,Femme,NaN,22,Bac+3,NaN,Non-concerné,NaN,Je me demande parfois ce que les autres pensen...,"L’anxiété, je suis souvent très stressée pour ...",...,"J’aimerai être plus motivée à travailler, et a...",Elle m’a apporté une confirmation dans mes cho...,Ma manière de faire des choix,"Mes pensées, ma nostalgie et le fait d’être bl...",NaN,Aucun,Etudiant,J'ai réalisé/je réalise une thérapie de soutien,NaN,NaN
1,14,Femme,NaN,20,Bac+3,NaN,Non-concerné,NaN,J’étais plus belle avant \n\nIl y a trop de tr...,"Stress, trop de choses à faire \nPression",...,Moins de trajet,/,/,/,NaN,Aucun,Etudiant,Je n'ai jamais réalisé de psychothérapie,NaN,NaN
2,17,Femme,NaN,21,Bac+3,NaN,Non-concerné,Dépression,"""Tout est de ma faute"" J'ai tendance à pas mal...","La tristesse, souvent reliée au sentiment de d...",...,J'aimerais ne plus être aussi dévalorisante en...,Je n'ai pas l'impression que ça m'ait apporté ...,Je ne sais pas,La force de mes pensées négatives,NaN,Dépression,Etudiant,J'ai réalisé/je réalise une psychothérapie mai...,NaN,Depression
3,19,Femme,NaN,21,Bac+4,Anxiété sociale,Non,"Dépression, burn-out",Je ne vais jamais réussir à trouver un boulot ...,"Anxiété, stress",...,Mon stress et ses effets sur mon corps,De l'écoute,La gestion de mon angoisse et d'autres petites...,"La peur de changer, la peur d'aller toujours m...",NaN,Anxiété sociale,Etudiant,"J'ai réalisé/je réalise une psychanalyse, J'ai...",Anxiety,Depression
4,20,Femme,NaN,21,Bac+3,NaN,Non-concerné,Dépression et trouble de la dépersonnalisation...,« Je ne serai jamais à la hauteur » je me reme...,NaN,...,Aller voir un(e) psychologue,Je suis aller dans une maison de l’adolescence...,NaN,Je n’ose pas demander à mes parents de payer p...,NaN,Autre,Etudiant,J'ai réalisé/je réalise une psychothérapie mai...,NaN,"Depression, Other psychiatric disorder"


In [ ]:
# Vérification du format général de la base nettoyée
df.shape, df.columns

Nous vérifions le nombre de lignes / colonnes et la liste des variables
présentes dans la base nettoyée.

## 3. Sauvegarde de la base nettoyée

La base nettoyée `df` est ensuite sauvegardée pour être réutilisée dans les 
notebooks d'analyse :

- `Analyse descriptive E2.ipynb`
- `BERTopic E2.ipynb`
- `Wordcloud.E2.ipynb`

In [ ]:
# Sauvegarde de la base nettoyée pour réutilisation ultérieure
df.to_csv("Data/Base1Base2_clean.csv", sep=",", index=False)